# Hiding in the Noise: Realistic GW Data with Sage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnarenraju/sage/blob/main/notebooks/colab/02_data_simulation.ipynb)

Gravitational-wave signals are buried under detector noise that is 10–100× larger in amplitude. This notebook shows how we go from raw LIGO strain to a whitened representation where signals become visible.

**What you will learn:**
- What real LIGO O3a strain data looks like
- How to estimate the noise power spectral density (PSD)
- How whitening reveals an injected signal
- How matched filtering extracts the SNR time series
- How multirate time-domain sampling compresses the data

**Runtime:** ~15 min on T4 (includes ~2 min GWOSC download)

## Setup

In [ ]:
import subprocess, sys
try:
    import sage
    print('Sage already installed.')
except ImportError:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nnarenraju/sage.git'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'sage/'], check=True)
    print('Sage installed.')


In [ ]:
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib.pyplot as plt
import torch

from gwpy.timeseries import TimeSeries
from pycbc.psd import aLIGOZeroDetHighPower
from pycbc.filter import matched_filter
from pycbc.types import FrequencySeries, TimeSeries as PyCBCTimeSeries
from sage.core.base_classes import BaseConfig, BaseDataConfig
from sage.core.config import register_configs
from sage.data.waveform.approximants.IMRPhenomD import IMRPhenomD
from sage.dsp.multirate_sampling import MultirateSampler, DyadicPyramidBinning

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


In [ ]:
class TutorialCFG:
    batch_size    = 32
    device        = device
    dtype         = torch.float32
    detectors     = ['H1', 'L1']
    do_point_estimate = []
    class_balance = 0.5
    clip_norm     = 1.0
    autocast      = False

class TutorialDataCFG:
    sample_rate                 = 2048.0
    signal_low_frequency_cutoff = 20.0
    sample_length_in_s          = 8.0
    padding_length_in_s         = 2.0

register_configs(BaseConfig(TutorialCFG()), BaseDataConfig(TutorialDataCFG()))
print('Configs registered.')


## 1. Download Real LIGO O3a Data

We fetch 256 seconds of open LIGO O3a strain from GWOSC. This takes ~2 minutes on a Colab session.

The chosen GPS window (1238782000–1238782256) is a quiet O3a segment — no known events.

In [ ]:
gps_start = 1238782000
gps_end   = gps_start + 256
fs        = 2048  # Hz

print(f'Fetching H1 data ({gps_start}–{gps_end}) ...')
strain_H1 = TimeSeries.fetch_open_data('H1', gps_start, gps_end, sample_rate=fs)
print(f'Fetching L1 data ({gps_start}–{gps_end}) ...')
strain_L1 = TimeSeries.fetch_open_data('L1', gps_start, gps_end, sample_rate=fs)
print(f'Downloaded: {len(strain_H1)/fs:.0f} s @ {fs} Hz per detector')


## 2. Raw Strain — Noise Dominates

Even 5 seconds of the raw detector output shows why direct visual inspection is hopeless — the noise amplitude is enormous compared to any realistic GW signal.

In [ ]:
t_H1 = strain_H1.times.value
x_H1 = strain_H1.value
t_L1 = strain_L1.times.value
x_L1 = strain_L1.value

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_H1[:5*fs] - t_H1[0], x_H1[:5*fs], lw=0.5)
axes[0].set_ylabel('Strain (H1)')
axes[1].plot(t_L1[:5*fs] - t_L1[0], x_L1[:5*fs], lw=0.5, color='C1')
axes[1].set_ylabel('Strain (L1)')
axes[1].set_xlabel('Time (s)')
fig.suptitle('Raw LIGO O3a strain — first 5 seconds')
plt.tight_layout()
plt.show()


## 3. Power Spectral Density Estimation

The noise PSD $S_n(f)$ describes the noise power per frequency bin. We estimate it via Welch's method (averaged periodograms over 4-second segments) and compare with the aLIGO design curve.

In [ ]:
from scipy.signal import welch

nperseg = 4 * fs   # 4-second Welch segments

f_welch, psd_H1 = welch(x_H1, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
_, psd_L1       = welch(x_L1, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)

# Design PSD for comparison
n_design = nperseg // 2 + 1
df_design = fs / nperseg
psd_design_obj = aLIGOZeroDetHighPower(n_design, df_design, 20.0)
psd_design = np.array(psd_design_obj.data[:])

fig, ax = plt.subplots(figsize=(10, 5))
mask = f_welch >= 20
ax.loglog(f_welch[mask], np.sqrt(psd_H1[mask]), label='H1 (O3a)', lw=1.5)
ax.loglog(f_welch[mask], np.sqrt(psd_L1[mask]), label='L1 (O3a)', lw=1.5)
ax.loglog(f_welch[mask], np.sqrt(psd_design[mask]), 'k--', label='aLIGO design', lw=1.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude spectral density (1/√Hz)')
ax.set_xlim(20, fs // 2)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Whitening

Whitening divides the frequency-domain strain by the noise ASD, making all frequencies equally loud on average. A whitened Gaussian noise segment has flat spectrum and unit variance per bin.

We use the Welch ASD estimated from the full 256-second segment.

In [ ]:
def whiten_segment(td_signal, psd_est, freq_est, target_fs, f_low=20.0):
    """Whiten a time-domain segment using a pre-estimated PSD."""
    n = len(td_signal)
    # FFT
    fd = np.fft.rfft(td_signal)
    freqs = np.fft.rfftfreq(n, d=1.0 / target_fs)
    # Interpolate PSD onto FFT frequency grid
    psd_interp = np.interp(freqs, freq_est, psd_est, left=np.inf, right=np.inf)
    asd_interp = np.sqrt(np.maximum(psd_interp, 1e-60))
    # Divide by ASD and taper below f_low
    fd_white = fd / asd_interp
    fd_white[freqs < f_low] = 0.0
    # Back to time domain
    return np.fft.irfft(fd_white, n=n)

# Use last 32 s of the segment for demonstration (avoid boundary effects)
n_demo = 32 * fs
x_demo_H1 = x_H1[-n_demo:]
x_demo_L1 = x_L1[-n_demo:]

white_H1 = whiten_segment(x_demo_H1, psd_H1, f_welch, fs)
white_L1 = whiten_segment(x_demo_L1, psd_L1, f_welch, fs)

t_demo = np.arange(n_demo) / fs

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_demo, white_H1, lw=0.5, label='H1 whitened')
axes[0].set_ylabel('Whitened strain (H1)')
axes[1].plot(t_demo, white_L1, lw=0.5, color='C1', label='L1 whitened')
axes[1].set_ylabel('Whitened strain (L1)')
axes[1].set_xlabel('Time (s)')
fig.suptitle('Whitened O3a noise — last 32 s of segment')
plt.tight_layout()
plt.show()


## 5. Signal Injection

We generate a 30+20 M☉ binary signal at optimal SNR 12 and inject it into the whitened noise at 25 seconds. After whitening, the signal should be visible as a brief amplitude increase.

In [ ]:
# Frequency grid matching the demo segment
f_l, f_u, del_f = 20.0, 1024.0, 1.0 / 32.0   # df = 1/32 Hz for 32-s segment
n_waveform = int(round((f_u - f_l) / del_f)) + 1
n_padded   = int(round(f_u / del_f)) + 1

params = torch.tensor(
    [[30., 20., 0., 0., 400., 0., 0., np.pi/4., 0.5, 1.2, 0.4]],
    dtype=torch.float64, device=device
)
f = (f_l + del_f * torch.arange(n_waveform, dtype=torch.float64, device=device)).unsqueeze(0).clone()
f_ref = torch.full((1, 1), f_l, dtype=torch.float64, device=device)

hp, hc = IMRPhenomD(f, f_ref)(params, reproduce_lal=True)
hp_np = hp.squeeze(0).cpu().numpy()  # (n_padded,) complex

# Compute optimal SNR with Welch PSD interpolated to del_f grid
freqs_fd = np.arange(n_padded) * del_f
psd_fd   = np.interp(freqs_fd, f_welch, psd_H1, left=np.inf, right=np.inf)
psd_fd[:int(f_l / del_f)] = np.inf
rho_sq = (4.0 / del_f) * np.sum(np.abs(hp_np)**2 / psd_fd)
rho_intrinsic = np.sqrt(rho_sq)
print(f'Intrinsic optimal SNR (H1 only): {rho_intrinsic:.1f}')

# Rescale to target SNR 12
target_snr = 12.0
scale = target_snr / rho_intrinsic
hp_scaled = hp_np * scale

# IFFT to time domain and crop to demo segment length
hp_td = np.fft.irfft(hp_scaled, n=n_demo)
hp_td_white = whiten_segment(hp_td, psd_H1, f_welch, fs)

# Inject at t_inj = 25 s (signal is in the last few seconds)
t_inj = 25
inject_idx = t_inj * fs
injected = white_H1.copy()
injected += np.roll(hp_td_white, inject_idx)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_demo, white_H1, lw=0.5, alpha=0.7, label='noise only')
axes[0].set_ylabel('H1 (whitened)')
axes[0].legend()
axes[1].plot(t_demo, injected, lw=0.5, color='C2', label=f'noise + signal (SNR {target_snr})')
axes[1].axvline(t_inj, color='r', lw=1, ls='--', label='injection time')
axes[1].set_ylabel('H1 (whitened + signal)')
axes[1].set_xlabel('Time (s)')
axes[1].legend()
fig.suptitle('Signal injection into whitened O3a noise')
plt.tight_layout()
plt.show()


## 6. Matched-Filter SNR Time Series

A matched filter correlates the data with the template waveform. It produces a peak at the signal arrival time — the height equals the optimal SNR. This is the basis of traditional GW searches.

In [ ]:
# Build PyCBC objects for matched_filter
df_mf = 1.0 / 32.0   # matches our segment
n_mf  = n_demo // 2 + 1

# Re-whiten injected data in FD
inj_fd = np.fft.rfft(injected)
psd_fd_mf = np.interp(np.arange(n_mf) * df_mf, f_welch, psd_H1, left=1e-40, right=1e-40)

data_pycbc = PyCBCTimeSeries(injected.astype(np.float64), delta_t=1.0/fs)

# Template (hp at 400 Mpc)
n_tmp = n_padded
template_fd = FrequencySeries(hp_np * scale, delta_f=del_f)
psd_pycbc = FrequencySeries(
    np.interp(np.arange(n_mf) * (1.0/32.0), freqs_fd, psd_fd, left=1e-40, right=1e-40),
    delta_f=1.0/32.0
)

snr_ts = matched_filter(template_fd, data_pycbc, psd=psd_pycbc, low_frequency_cutoff=f_l)
snr_abs = abs(snr_ts)

t_snr = snr_ts.sample_times.numpy()
t_snr -= t_snr[0]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_snr, snr_abs.numpy(), lw=0.8, label='|SNR(t)|')
ax.axvline(t_inj, color='r', lw=1.5, ls='--', label=f'injection at t={t_inj}s')
ax.axhline(target_snr, color='k', lw=1, ls=':', label=f'target SNR = {target_snr}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Matched-filter SNR')
ax.set_title('Matched-filter SNR time series')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Peak SNR: {float(snr_abs.max()):.2f} (target: {target_snr})')


## 7. Multirate Sampling Compression

CBC signals sweep from low to high frequency. Early in the inspiral the signal occupies only a narrow band around 20–50 Hz. Sage's multirate sampler decimates each time slice to the minimum sample rate needed for the GW content at that moment, saving ~10× in data volume with no information loss.

In [ ]:
# DyadicPyramidBinning needs mass and tc bounds from the prior
param_bounds = {
    'mass1': (7.0, 50.0),  # prior mass range
    'tc':    (5.0, 7.0),   # coalescence time within the segment (s)
}

binning = DyadicPyramidBinning(param_bounds, lowest_allowed_fs=64.0)
mrsampler = MultirateSampler(binning)

print(f'Bin layout: {len(binning.detailed_bins)} bins')
for b in binning.detailed_bins:
    dur = (b[1] - b[0]) / 2048.0
    print(f'  [{b[0]:5d}–{b[1]:5d}] samples  →  fs = {b[2]} Hz  ({dur:.3f} s)')


In [ ]:
# Apply multirate sampling to the whitened demo segment
# Input shape: (B, D, L)
x_td = torch.tensor(
    np.stack([white_H1, white_L1], axis=0)[np.newaxis, ...],  # (1, 2, n_demo)
    dtype=torch.float32, device=device
)

# Crop to match the expected segment length from config
seg_len = int(TutorialDataCFG.sample_length_in_s * TutorialDataCFG.sample_rate)
x_td_seg = x_td[:, :, :seg_len]  # (1, 2, seg_len)

with torch.no_grad():
    x_compressed = mrsampler(x_td_seg)  # (1, 2, L_compressed)

original_len   = x_td_seg.shape[-1]
compressed_len = x_compressed.shape[-1]
ratio = original_len / compressed_len

print(f'Original length:   {original_len:6d} samples  ({original_len / 2048:.2f} s at 2048 Hz)')
print(f'Compressed length: {compressed_len:6d} samples')
print(f'Compression ratio: {ratio:.1f}×')
